### CNN for Classifying the MNIST Dataset on GPU
#### Also plot loss and accuracy curves and display some sample images along with classification results

Implementation Steps:
* Import Libraries: Import necessary libraries including PyTorch and matplotlib.
* Define CNN Model: Create a simple CNN model.
* Load Data: Load and preprocess the MNIST dataset.
* Train Model: Train the model and record the loss and accuracy.
* Plot Results: Plot the loss and accuracy curves.
* Display Sample Predictions: Show some sample images along with their predicted labels.



---
## Background: What Is a CNN?

A **Convolutional Neural Network (CNN)** is a type of deep learning model designed specifically for image analysis. Instead of treating an image as a flat list of numbers, a CNN scans the image with small filters to detect patterns like edges, curves, and textures — then combines those patterns to recognise objects or classes.

### How a CNN processes an image step by step:

```
Input image (28×28 pixels, 1 channel — grayscale)
        ↓
Conv Layer 1  → apply 32 filters (3×3) → detect edges and simple shapes
        ↓
ReLU          → set all negative values to 0 (adds non-linearity)
        ↓
MaxPool       → shrink 28×28 → 14×14 (keep strongest features)
        ↓
Conv Layer 2  → apply 64 filters (3×3) → detect complex patterns
        ↓
ReLU
        ↓
MaxPool       → shrink 14×14 → 7×7
        ↓
Flatten       → convert 7×7×64 feature map into a 1D vector (3136 values)
        ↓
Fully Connected Layer (128 neurons)
        ↓
Output Layer  → 10 neurons, one per digit class (0–9)
```

### Key terms to know:

| Term | Meaning |
|---|---|
| **Filter/Kernel** | A small 3×3 matrix of weights the model learns to detect one feature |
| **Feature map** | The output after applying one filter to the image |
| **ReLU** | Activation function — passes positive values, blocks negative ones |
| **MaxPooling** | Downsamples the image by keeping only the maximum value in each region |
| **Flatten** | Converts a 2D feature map into a 1D vector for the dense layers |
| **Loss** | How wrong the model's prediction was — we want this to decrease |
| **Optimizer (Adam)** | Algorithm that updates the model's weights to reduce loss |

### About the MNIST dataset

MNIST is a classic benchmark dataset of **70,000 grayscale images** of handwritten digits (0–9). Each image is 28×28 pixels. 60,000 images are used for training; 10,000 for testing. It is one of the most widely used datasets in machine learning education.

> **Before running any code:** Enable the GPU.
> Go to **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**
> Running on GPU makes training ~10× faster than CPU.


---
## Step 1: Import Libraries

We use **PyTorch** as our deep learning framework — it handles building the model, computing gradients, and running everything on the GPU. We also import **torchvision** (which provides the MNIST dataset and image transforms) and **matplotlib** (for plotting).

Run this cell first before anything else.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')


ModuleNotFoundError: No module named 'matplotlib'

---
## Step 2: Define the CNN Model

We define our CNN as a Python class. Every model in PyTorch has two parts:
- `__init__`: defines all the layers
- `forward`: defines how data flows through those layers

### Layer-by-layer explanation:

| Layer | Code | What it does |
|---|---|---|
| Conv1 | `Conv2d(1, 32, 3)` | Takes 1 input channel (grayscale), applies 32 filters of size 3×3 |
| Conv2 | `Conv2d(32, 64, 3)` | Takes 32 channels from conv1, applies 64 filters |
| Pool  | `MaxPool2d(2, 2)` | Keeps max value in each 2×2 region — halves the spatial size |
| FC1   | `Linear(64×7×7, 128)` | Fully connected layer: 3136 inputs → 128 outputs |
| FC2   | `Linear(128, 10)` | Output layer: 128 inputs → 10 class scores (one per digit) |

The formula in the comment `z=(H - f + 2p)/s + 1` calculates the output size of a convolution:
- H = input height, f = filter size, p = padding, s = stride
- E.g. for conv1: (28 - 3 + 2×1)/1 + 1 = 28 → after pooling: 14×14


In [ ]:
# 1. Define the CNN Model
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1) # 28 x 28 --> z=(H - f + 2p)/s + 1 ==> 14 x 14
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1) # 7 x 7
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.fc1 = nn.Linear(64 * 7 * 7, 128) # 28//2//2
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7) # flatten
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Quick check: print the model structure
print(SimpleCNN())


---
## Step 3: Load and Preprocess the Data

Before feeding images to the CNN, we apply two **transforms**:

1. **`ToTensor()`** — Converts a PIL image (values 0–255) to a PyTorch tensor (values 0.0–1.0). It also changes the axis order from (H, W, C) to (C, H, W) as PyTorch expects.

2. **`Normalize((0.1307,), (0.3081,))`** — Subtracts the mean (0.1307) and divides by the standard deviation (0.3081) of the MNIST dataset. This rescales pixel values to have approximately zero mean and unit variance, which helps the model train faster and more stably.

A **DataLoader** wraps the dataset and serves images in random mini-batches during training. We use `batch_size=64` for training (64 images per update step) and `batch_size=1000` for testing (just for speed — we are not updating weights during evaluation).


In [ ]:
# 2. Load Data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

print(f'Training samples: {len(train_dataset):,}')
print(f'Test samples:     {len(test_dataset):,}')
print(f'Training batches: {len(train_loader)} (at batch size 64)')
print(f'\nOne image shape: {train_dataset[0][0].shape}  → (channels, height, width)')


---
## Step 4: Train the Model

Training repeats the following steps for every **epoch** (one full pass through the training data):

1. **Forward pass** — feed a batch of images through the model to get predictions
2. **Compute loss** — measure how wrong the predictions are using `CrossEntropyLoss`
3. **Backward pass (backpropagation)** — calculate how much each weight contributed to the error
4. **Update weights** — the optimizer (`Adam`) adjusts all weights slightly to reduce the loss
5. After each epoch, **evaluate on the test set** to see how well the model generalises

### What to watch:
- **Loss** should decrease each epoch — the model is making fewer errors
- **Train Accuracy** should increase — the model is learning the training data
- If train accuracy is much higher than test accuracy, the model may be **overfitting**
  (memorising the training data rather than learning general patterns)

⏱️ With GPU enabled this should take about **2–3 minutes** for 10 epochs.


In [ ]:
# 3. Train Model
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10
train_losses = []
train_accuracies = []
test_accuracies = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_losses.append(running_loss / len(train_loader))
    train_accuracies.append(100 * correct / total)

    # Evaluate on test set
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_accuracies.append(100 * correct / total)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}, Train Accuracy: {100 * correct/total:.2f}%')


---
## Step 5: Plot Loss and Accuracy Curves

These two plots are the standard way to evaluate how well training went:

- **Loss curve** — should steadily decrease. A flattening curve means the model has stopped improving.
- **Accuracy curves** — training and test accuracy should both rise and stay close to each other. A large gap between them indicates overfitting.

Take a screenshot of these plots — you will need them to answer the questions at the end.


In [ ]:
# 4. Plot Loss and Accuracy Curves
plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss Curve')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Training Accuracy')
plt.plot(test_accuracies, label='Test Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy Curve')
plt.legend()

plt.tight_layout()
plt.savefig('cnn_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Final Test Accuracy:', round(test_accuracies[-1], 2), '%')


---
## Step 6: Display Sample Predictions

Now we pick random images from the test set and ask the model to predict their digit class. For each image we show:
- The actual handwritten digit
- The **true label** (what the digit actually is)
- The **predicted label** (what the model thinks it is)

Correct predictions are shown in **green**, wrong ones in **red**. Look carefully at any red titles — they reveal which digits the model finds most confusing.


In [ ]:
# 5. Display Sample Predictions
model.eval()
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)

# Move to CPU for plotting
images = images.cpu().numpy()
labels = labels.cpu().numpy()
predicted = predicted.cpu().numpy()

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle('Sample Predictions — Green = Correct, Red = Wrong',
             fontsize=13, fontweight='bold')

for i, ax in enumerate(axes.flatten()):
    ax.imshow(images[i].squeeze(), cmap='gray')
    color = 'green' if predicted[i] == labels[i] else 'red'
    ax.set_title(f'T:{labels[i]}  P:{predicted[i]}', fontsize=9,
                 color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('cnn_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('T = True label | P = Predicted label')


---
## Reflection Questions

*Double-click this cell to type your answers directly into the notebook.*

---

**Q1.** After training, what was your final **test accuracy** (from the last epoch printed in Step 4)? Is it higher or lower than the training accuracy? What does this difference (or lack of difference) tell you?

> *Your answer:*

---

**Q2.** Look at your **Loss Curve** from Step 5. Describe the shape of the curve. What would it mean if the loss curve flattened out completely after just 2 epochs?

> *Your answer:*

---

**Q3.** Look at your **sample predictions** from Step 6. Find at least one image that was predicted **incorrectly** (red title). Describe what the true digit is and what the model predicted instead. Why do you think the model made that mistake?

> *Your answer:*

---

**Q4.** In the CNN definition (Step 2), the first convolutional layer uses `padding=1`. What would happen to the size of the feature map if you removed the padding (`padding=0`) and kept everything else the same?

*Hint: use the formula from the comment in the code: z = (H − f + 2p) / s + 1*

> *Your answer:*

---

**Q5.** The training loop calls `optimizer.zero_grad()` at the start of every batch. What would happen if you forgot to include this line?

> *Your answer:*

---

**Q6.** This CNN was trained on **handwritten digits** (MNIST). If you wanted to adapt this same model to classify **Sentinel-2 satellite image patches** into 10 land cover classes (like EuroSAT), what would you need to change in the model definition and why?

> *Your answer:*
